In [2]:
import os
from dotenv import load_dotenv


load_dotenv()
API_KEY = os.getenv("GROQ_API_KEY")
# API_KEY

In [3]:
os.environ["GROQ_API_KEY"] = API_KEY

In [183]:
from langchain.document_loaders import PyPDFLoader

In [184]:
file_path = "data/sample2.pdf"
loader = PyPDFLoader(file_path)

In [185]:
data = loader.load()
data
# len(data)

[Document(metadata={'source': 'data/sample2.pdf', 'page': 0}, page_content='arXiv:chao-dyn/9710005v2  17 Oct 1997October 1997\nchao-dyn/9710005\nTOWARDS THE GLOBAL:\nCOMPLEXITY, TOPOLOGY AND CHAOS\nIN MODELLING, SIMULATION AND COMPUTATION\nDavid A. Meyer\nCenter for Social Computation, Institute for Physical Scie nces, and\nProject in Geometry and Physics\nDepartment of Mathematics\nUniversity of California/San Diego\nLa Jolla, CA 92093-0112\ndmeyer@chonji.ucsd.edu\nABSTRACT\nTopological eﬀects produce chaos in multiagent simulation and distributed computation.\nWe explain this result by developing three themes concernin g complex systems in the\nnatural and social sciences: ( i) Pragmatically, a system is complex when it is represented\neﬃciently by diﬀerent models at diﬀerent scales. ( ii) Nontrivial topology, identiﬁable as\nwe scale towards the global, inducescomplexity in this sense. ( iii) Complex systems with\nnontrivial topology are typically chaotic.\nJournal of Economic Liter

In [191]:
question_gen = ""
for page in data:
    question_gen = question_gen + page.page_content

In [192]:
from langchain.text_splitter import TokenTextSplitter
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [193]:
splitter_gen = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

In [194]:
chunks_generation = splitter_gen.split_text(question_gen)
# chunks_question_generation

In [143]:
len(chunks_generation)

18

In [195]:
from langchain.docstore.document import Document

In [186]:
doc_generation = [Document(page_content = t) for t in chunks_generation ]

In [145]:
documnet_splitter_template = RecursiveCharacterTextSplitter(
    chunk_size=2000,  # number of characters
    chunk_overlap=200
)

In [146]:
docs =  documnet_splitter_template.split_documents(doc_generation)

In [147]:
len(docs)

18

In [75]:
from langchain_groq import ChatGroq

In [112]:
llm = ChatGroq(
    model="llama-3.1-8b-instant", 
   temperature=0,
    max_tokens=512,
   
)

In [118]:
from langchain.prompts import PromptTemplate

In [148]:


prompt = """
Write a concise summary of the following text.
Focus on key ideas and conclusions.

TEXT:
{text}
"""


In [149]:
SUMMARY_PROMPT = PromptTemplate(
    template=prompt,
    input_variables=["text"]
)

In [127]:
from langchain.chains.summarize import load_summarize_chain

In [176]:
from langchain_core.output_parsers import StrOutputParser

In [177]:
# chain = load_summarize_chain(
#     llm,
#     chain_type="map_reduce",  # best for long PDFs
#     map_prompt=SUMMARY_PROMPT
# )
map_chain = SUMMARY_PROMPT | llm | StrOutputParser()


In [154]:
from langchain_core.runnables import RunnableConfig

config = RunnableConfig(
    max_concurrency=2  # start with 1 or 2
)

In [165]:
# summary = chain.invoke(docs)
# chunk_summaries = map_chain.invoke({
#     "text": docs[0].page_content
# })

chunk_summaries = []

for doc in docs:
    summary = map_chain.invoke({
        "text": doc.page_content
    })
    chunk_summaries.append(summary)

 

In [ ]:
print(chunk_summaries)

In [169]:
COMBINE_THIS_PROMPT = """
You are a professional summarizer.

Combine the following summaries into ONE concise summary.
Remove repetition and keep the most important ideas.

SUMMARIES:
{text}
"""

In [170]:
COMBINE_PROMPT = PromptTemplate(
    template=COMBINE_THIS_PROMPT,
    input_variables=["text"]
)

In [196]:
combine_chain = COMBINE_PROMPT | llm | StrOutputParser()

In [197]:
final_summary = combine_chain.invoke({
    "text": "\n\n".join(chunk_summaries)
})

# print(final_summary)

In [198]:
final_summary = final_summary.split("\n")
final_summary = [q.strip() for q in final_summary if q.strip()]

In [199]:
print(final_summary)

['After combining the provided summaries, I have extracted the key ideas and conclusions into a concise summary:', '**Complexity in Systems**', 'Complex systems exhibit complexity due to non-trivial topology, which leads to chaotic behavior. This complexity arises from the multiscale nature of the system, with different representations at different scales leading to increased complexity. Topology plays a crucial role in inducing new models at more global scales.', '**Pragmatic Definition of Complexity**', 'A system is considered complex if it can be efficiently represented by different models at different scales. Non-trivial topology induces complexity, and systems with non-trivial topology are prone to chaotic behavior.', '**Examples of Complex Systems**', '1. **Gravitational Simulations**: Large-scale states can be described by simpler models, such as total mass and average position, leading to "simplicity" at larger scales.', '2. **Quantum Gravity and Fluid Mechanics**: Non-trivial 